In [ ]:
# Getting all files:
import glob

data_files = glob.glob("../data/*/*.pdf")
print("Number of files found:", len(data_files))

file_path_to_id = {file_path: file_path.split("/")[-1] for file_path in data_files}
print("File path to ID mapping:", file_path_to_id)

In [ ]:
# Add syspath for module imports
import sys

sys.path.append("..")

# Example: Parse a PDF and inspect per-page structure
from modules import pdf_loader

IMAGES_PATH = "../data/images/"

# Adjust path to a real file in your `data/` tree before running
example_pdf = data_files[0]
res = pdf_loader.parse_pdf(example_pdf, extract_images=True, images_dir=IMAGES_PATH)
print("Parsed file:", res["file_path"])
print("Pages parsed:", len(res["pages"]))
for p in res["pages"][:3]:
    print(
        f"--- Page {p['page_number']} -- text length: {len(p['text'])} | tables: {len(p['tables'])} | images: {len(p['images'])}"
    )

In [ ]:
res["pages"][0]

# Parsing all the documents: 

In [ ]:
from tqdm import tqdm

all_parsed = []
for fp in tqdm(data_files):
    parsed = pdf_loader.parse_pdf(fp, extract_images=False, images_dir=None, doc_id=file_path_to_id[fp])
    all_parsed.extend(parsed["pages"])

In [ ]:
all_parsed[0]

In [ ]:
# Dump parsed data to a file for later use
import joblib

# fields_to_save = ["doc_id", "page_number", "text", "tables"]
# all_parsed = [{k: page[k] for k in fields_to_save} for page in all_parsed]
# joblib.dump(all_parsed, "../data/parsed/all_parsed_pages_v1.joblib")
all_parsed = joblib.load("../data/parsed/all_parsed_pages_v1.joblib")

In [ ]:
print("Total pages parsed from all documents:", len(all_parsed))

# Initial search index and search: 

In [ ]:
# Add syspath for module imports
import sys

sys.path.append("..")

from modules.tantivy_index import TantivyIndex

idx = TantivyIndex(index_dir=None, optional_fields=[])
idx.add_documents(all_parsed)

res = idx.search("Яка мета гри го?", top_k=5, include_snippets=False)
print(res)

# Testing search on dev dataset: 

In [ ]:
import numpy as np


def recall_at_k(results, ground_truth, k):
    """Compute Recall@K for a single query."""
    retrieved_ids = [doc["doc_id"] for doc in results[:k]]
    return int(ground_truth in retrieved_ids)


def page_score_np(page_pred, page_true, n_pages, d):
    """Compute the page score using numpy."""
    score = (1 - np.abs(page_pred - page_true) / n_pages) * d
    return score

In [ ]:
import pandas as pd

dev_set = pd.read_csv("../data/dev_questions.csv")
display(dev_set.head())

In [ ]:
from tqdm import tqdm

questions = dev_set["Question"].tolist()
results = []
for q in tqdm(questions):
    res = idx.search(q, top_k=10, include_snippets=False)
    results.append(res)

dev_set["retrieved_docs"] = results

In [ ]:
# Calculating metrics:
k_values = [1, 3, 5]
for k in k_values:
    dev_set[f"recall_at_{k}"] = dev_set.apply(lambda row: recall_at_k(row["retrieved_docs"], row["Doc_ID"], k), axis=1)
    print(f"Recall@{k}: {dev_set[f'recall_at_{k}'].mean():.4f}")

# Page score calculation:
dev_set["correct_doc"] = dev_set.apply(
    lambda row: 1 if row["retrieved_docs"][0]["doc_id"] == row["Doc_ID"] else 0, axis=1
)
dev_set["page_score"] = dev_set.apply(
    lambda row: page_score_np(
        row["retrieved_docs"][0]["page_number"],
        row["Page_Num"],
        row["N_Pages"],
        row["correct_doc"],
    ),
    axis=1,
)
print(f"Average Page Score: {dev_set['page_score'].mean():.4f}")

In [ ]:
idd = 3
print(dev_set.iloc[idd]["Question"])
print(dev_set.iloc[idd]["Doc_ID"])
dev_set.iloc[idd]["retrieved_docs"]